# AION-U1 Pilot-Training (Colab T4)

Zwei Läufe: Training und danach dieselbe Messung, die auch die Lehrer erfasst.
Ergebnis liegt in `reports/eval-pilot.json`; der Adapter in `artifacts/sft/pilot_open/final_model/`.

In [ ]:
# 1) Repo
!git clone -q https://github.com/Reznap87/aion-u1.git
%cd aion-u1
!pip install -q -e '.[train]'
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# 2) GPU-Praezision: die T4 (sm_75) unterstuetzt kein bf16.
#    `fp16: true` allein reicht nicht — `dtype: auto` laedt das Modell
#    in bfloat16 (Qwen-Konfiguration) und bricht auf sm_75 ab.
import pathlib
p = pathlib.Path('training/configs/sft_lora_open.yaml')
t = p.read_text(encoding='utf-8').replace('fp16: false', 'fp16: true')
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8:
    t = t.replace('dtype: auto', 'dtype: float16')
    print('sm_75 ohne bf16 -> dtype float16')
else:
    print('bf16 wird unterstuetzt, dtype bleibt', 'auto')
p.write_text(t, encoding='utf-8')
print('fp16 aktiv:', 'fp16: true' in p.read_text(encoding='utf-8'))


In [ ]:
# 3) Training (58 Trainingssaetze, LoRA r=16)
!aion-u1-train-sft --config training/configs/sft_lora_open.yaml
import pathlib
if not pathlib.Path('artifacts/sft/pilot_open/final_model/adapter_model.safetensors').is_file():
    raise SystemExit('Kein Adapter erzeugt — siehe Fehlermeldung von Zelle 3')
print('Adapter liegt vor:', pathlib.Path('artifacts/sft/pilot_open/final_model').resolve())


In [ ]:
# 4) Baseline (gleiche Suite, ohne Adapter) — fuer den Vergleich
!aion-u1-evaluate-model --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --suite evaluations/capability_suite/benchmark_hard_v1.yaml \
  --device cuda --output reports/eval-baseline.json

In [ ]:
# 5) Messung des Adapter-Modells (harte Suite, dieselbe Aggregation)
!aion-u1-evaluate-model --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --adapter artifacts/sft/pilot_open/final_model \
  --suite evaluations/capability_suite/benchmark_hard_v1.yaml \
  --device cuda --output reports/eval-pilot.json

In [ ]:
# 6) Ergebnis lesen
import json
for name in ('eval-baseline', 'eval-pilot'):
    try:
        d = json.load(open(f'reports/{name}.json', encoding='utf-8'))
        s = list(d['scores'].values())[0]
        print(f"{name:14} index={s.get('overall_score')} gemessen={s.get('measured_score')} coverage={s.get('coverage')}")
    except FileNotFoundError:
        print(name, 'fehlt')
print('Zielmarke GOAL.md: 0.847 (beste Lehrer, harte Suite)')

In [ ]:
# 7) Artefakte sichern (Adapter herunterladen, bevor die Session endet)
import pathlib
if not pathlib.Path('artifacts/sft/pilot_open').is_dir():
    raise SystemExit('artifacts/sft/pilot_open fehlt — Training (Zelle 3) pruefen')
!zip -q -r sft_pilot_open.zip artifacts/sft/pilot_open
from google.colab import files
files.download('sft_pilot_open.zip')